# P2 · N0 — Environment, Data Check, and Splits**Paper 2 — MARQ-Bench: Machine-Authored Data Quality**This notebook does four things and nothing else:1. Mounts Drive and imports the `llmauth_*` modules2. Finds your four data files and records their SHA-256 hashes3. Loads each corpus **safely** (see the `None` warning below)4. Builds the 20% authoring / 80% evaluation split for each corpus and saves the manifests**Resumability:** every expensive step is checkpointed to Drive. If Colabdisconnects, just re-run the notebook top to bottom — completed steps areskipped. To redo a step, delete its file from `checkpoints/` and re-run.**Runtime:** a few minutes. No GPU needed. No API keys needed.

## 1 · Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · PathsAdjust `ROOT` only if you renamed the project folder. `MODULES` points atwherever the `llmauth_*.py` files live — currently the `Notebooks` subfolder.

In [ ]:
import sys, os
from pathlib import Path
ROOT        = Path('/content/drive/MyDrive/Paper2_RuleAuthorship')
MODULES     = ROOT / 'Notebooks'          # where the llmauth_*.py files are
CHECKPOINTS = ROOT / 'checkpoints'
ARTIFACTS   = ROOT / 'artifacts'
for d in (CHECKPOINTS, ARTIFACTS):
    d.mkdir(parents=True, exist_ok=True)
assert ROOT.exists(), f'project folder not found: {ROOT}'
assert MODULES.exists(), f'modules folder not found: {MODULES}'
if str(MODULES) not in sys.path:
    sys.path.insert(0, str(MODULES))
print('ROOT       ', ROOT)
print('MODULES    ', MODULES)
print('CHECKPOINTS', CHECKPOINTS)

ROOT        /content/drive/MyDrive/Paper2_RuleAuthorship
MODULES     /content/drive/MyDrive/Paper2_RuleAuthorship/Notebooks
CHECKPOINTS /content/drive/MyDrive/Paper2_RuleAuthorship/checkpoints


## 3 · Install and import

In [ ]:
%pip install -q pyarrow

import llmauth_census as C
import llmauth_prompts as P
import llmauth_checkpoint as CK

print('census   ', C.CENSUS_VERSION)
print('prompts  ', P.PROMPT_VERSION)
print('checkpt  ', CK.CHECKPOINT_VERSION)
print('modules  ',MODULES)
print('Registered corpora:', list(C.CORPUS_READ_CONFIG))

census    1.0.0
prompts   1.0.0
checkpt   1.0.0
modules   /content/drive/MyDrive/Paper2_RuleAuthorship/Notebooks
Registered corpora: ['bank_marketing', 'diabetes_130us', 'online_retail_ii', 'nyc_tlc_yellow']


## 4 · Find the data filesThis searches the whole project folder rather than assuming a layout, andtells you exactly what it found. If a corpus shows `MISSING`, fix the filenameor path and re-run this cell.

In [ ]:
import hashlib

# filename fragments -> corpus id. Adjust if your filenames differ.
PATTERNS = {
    'bank_marketing':   ['bankfull', 'bank-full', 'bank_full'],
    'diabetes_130us':   ['diabetic_data', 'diabetes'],
    'online_retail_ii': ['online_retail', 'online retail', 'retail'],
    'nyc_tlc_yellow':   ['yellow_tripdata'],
}

# Search for files
all_files = [p for p in ROOT.rglob('*')
             if p.is_file() and p.suffix.lower() in ('.csv', '.parquet', '.xlsx')]

DATA_PATHS = {}
for corpus, frags in PATTERNS.items():
    hits = [p for p in all_files
            if any(f in p.name.lower() for f in frags)]
    # prefer the largest match
    hits.sort(key=lambda p: p.stat().st_size, reverse=True)
    if hits:
        DATA_PATHS[corpus] = hits[0]

print(f'{"corpus":<18} {"status":<9} {"size":>12}  file')
print('-' * 78)
for corpus in PATTERNS:
    p = DATA_PATHS.get(corpus)
    if p:
        print(f'{corpus:<18} {"FOUND":<9} {p.stat().st_size:>12,}  {p.relative_to(ROOT)}')
    else:
        print(f'{corpus:<18} {"MISSING":<9} {"-":>12}  --')

missing = [c for c in PATTERNS if c not in DATA_PATHS]
if missing:
    print(f'\n!! Missing: {missing}')
    print('   Upload them to your Drive folder, or edit PATTERNS above to match your filenames.')

corpus             status            size  file
------------------------------------------------------------------------------
bank_marketing     FOUND        4,610,348  data/bank-full.csv
diabetes_130us     FOUND       19,159,383  data/diabetic_data.csv
online_retail_ii   FOUND       45,622,278  data/online_retail_II.xlsx
nyc_tlc_yellow     FOUND       69,699,174  data/c4_nyc_tlc/yellow_tripdata_2026-05.parquet


## 5 · Record file hashesThese go in the reproducibility artifact. If a source file is ever silentlyreplaced upstream, these hashes are what prove which version you used.

In [ ]:
import hashlib

# Safety check: Ensure DATA_PATHS was defined in the previous section
if 'DATA_PATHS' not in globals():
    raise NameError("DATA_PATHS is not defined. Please run the cell in 'Section 4 · Find the data files' (cell LGj7ecXNaIPY) before running this cell.")

ckpt = CK.Checkpoint(CHECKPOINTS)

def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for blk in iter(lambda: fh.read(chunk), b''):
            h.update(blk)
    return h.hexdigest()

def build_hashes():
    return {c: {'path': str(p.relative_to(ROOT)),
                'bytes': p.stat().st_size,
                'sha256': _sha256(p)}
            for c, p in sorted(DATA_PATHS.items())}

file_hashes, _ = ckpt.step('source_file_hashes', 'json', build_hashes,
                           code_version=C.CENSUS_VERSION)

for c, h in file_hashes.items():
    print(f'{c:<18} {h["sha256"][:16]}...  {h["bytes"]:>12,} bytes')

  [cached] source_file_hashes.json  (written 2026-08-10T13:07:52+00:00)
bank_marketing     d1513ec63b385506...     4,610,348 bytes
diabetes_130us     0689e7ec031237dc...    19,159,383 bytes
nyc_tlc_yellow     9aa5a1609e2bf07d...    69,699,174 bytes
online_retail_ii   bcbe73b35f5b7bab...    45,622,278 bytes


## 6 · Load each corpus safely`load_corpus` uses an explicit per-corpus read configuration and **refuses** toguess.> **Why this matters.** pandas' default NA list contains the string `"None"`.> A plain `pd.read_csv` on `diabetic_data.csv` converts all 96,420 occurrences> of `max_glu_serum = "None"` into `NaN`. That value is *valid* — it means the> lab test was not administered — and losing it would silently destroy one of> this paper's central findings. Never call `pd.read_csv` on these files> directly.

In [ ]:
import pandas as pd
import sys, os, importlib
from pathlib import Path

# 1. Path Injection: ensure custom modules are visible
ROOT = Path('/content/drive/MyDrive/Paper2_RuleAuthorship')
MODULES = ROOT / 'Notebooks'
if str(MODULES) not in sys.path:
    sys.path.insert(0, str(MODULES))

# 2. Strict Module Loading
import llmauth_census as C
importlib.reload(C)

# Ensure we are using the version that supports full retail loading
assert 'expected_rows' in C.CORPUS_READ_CONFIG['online_retail_ii'], \
    'Please upload the updated llmauth_census.py (v1.1.0) to your Notebooks folder.'

# 3. Load all corpora strictly through C.load_corpus
corpora = {}
print(f'{"corpus":<18} {"rows":>10} x {"cols":>2}  {"notes"}')
print('-' * 60)

for corpus, path in DATA_PATHS.items():
    df, prov = C.load_corpus(corpus, path)
    corpora[corpus] = df

    sheets = prov.get('sheet_rows')
    note = f"sheets={sheets}" if sheets else ""
    print(f'{corpus:<18} {df.shape[0]:>10,} x {df.shape[1]:>2}  {note}')

# 4. Integrity Assertions
d = corpora['diabetes_130us']
assert (d.max_glu_serum == 'None').sum() == 96420, "Diabetes 'None' sentinel lost"
assert (d.weight == '?').sum() == 98569, "Diabetes '?' sentinel lost"

b = corpora['bank_marketing']
assert (b.pdays == -1).sum() == 36954, "Bank 'pdays' sentinel lost"

r = corpora['online_retail_ii']
assert len(r) == 1_067_371, f"Retail truncated: {len(r):,} rows"

print("\nOK: All corpora loaded safely with integrity sentinels verified.")

# Checkpoint instance for subsequent sections
ckpt = CK.Checkpoint(CHECKPOINTS)

corpus                   rows x cols  notes
------------------------------------------------------------
bank_marketing         45,211 x 17  
diabetes_130us        101,766 x 50  
online_retail_ii    1,067,371 x  8  sheets={'Year 2009-2010': 525461, 'Year 2010-2011': 541910}
nyc_tlc_yellow      4,090,836 x 20  

OK: All corpora loaded safely with integrity sentinels verified.


## 7 · Build the authoring / evaluation splitsRules are authored from a profile of the **20% authoring split**. Every metricis computed on the disjoint **80% evaluation split**.Without this separation the census would describe the same rows the rules arelater scored on — leakage a reviewer will find. C4 is subsampled to 1M rowsfirst, with a pinned seed.The split is deterministic: same seed, same rows, every time.

In [ ]:
SUBSAMPLE = {'nyc_tlc_yellow': 1_000_000}   # C4 only
splits, manifests = {}, {}

for corpus, df in corpora.items():
    def build(corpus=corpus, df=df):
        auth, ev, man = C.make_split(
            df, corpus, subsample_rows=SUBSAMPLE.get(corpus))
        return {'manifest': man.__dict__,
                'authoring_index': auth.index.tolist(),
                'evaluation_index': ev.index.tolist()}

    payload, cached = ckpt.step(f'split_{corpus}', 'json', build,
                               code_version=C.CENSUS_VERSION)
    manifests[corpus] = payload['manifest']
    splits[corpus] = (
        df.loc[payload['authoring_index']],
        df.loc[payload['evaluation_index']],
    )

print()
print(f'{"corpus":<18} {"authoring":>11} {"evaluation":>11}  {"auth index sha":<18}')
print('-' * 66)
for corpus, m in manifests.items():
    print(f'{corpus:<18} {m["authoring_rows"]:>11,} {m["evaluation_rows"]:>11,}  '
          f'{m["authoring_index_sha256"][:16]}...')

  [cached] split_bank_marketing.json  (written 2026-08-08T03:39:10+00:00)
  [cached] split_diabetes_130us.json  (written 2026-08-08T03:39:11+00:00)
  [cached] split_online_retail_ii.json  (written 2026-08-08T14:34:08+00:00)
  [cached] split_nyc_tlc_yellow.json  (written 2026-08-08T03:39:14+00:00)

corpus               authoring  evaluation  auth index sha    
------------------------------------------------------------------
bank_marketing           9,042      36,169  357727425d51297e...
diabetes_130us          20,353      81,413  d6549d2c0a568fce...
online_retail_ii       213,474     853,897  f42192148065ed4e...
nyc_tlc_yellow         200,000     800,000  949163fc299d544d...


## 8 · Verify the splits

In [ ]:
ok = True
for corpus, (auth, ev) in splits.items():
    overlap = len(set(auth.index) & set(ev.index))
    total   = len(auth) + len(ev)
    frac    = len(auth) / total
    source  = manifests[corpus]['subsample_rows'] or manifests[corpus]['source_rows']

    good = (overlap == 0) and (total == source) and abs(frac - 0.20) < 0.001
    ok &= good

    print(f'{"OK " if good else "!! "}{corpus:<18} overlap={overlap}  '
          f'total={total:,}/{source:,}  authoring={frac:.4f}')

print('\nAll splits valid.' if ok else '\n!! SPLIT PROBLEM — do not proceed.')

OK bank_marketing     overlap=0  total=45,211/45,211  authoring=0.2000
OK diabetes_130us     overlap=0  total=101,766/101,766  authoring=0.2000
OK online_retail_ii   overlap=0  total=1,067,371/1,067,371  authoring=0.2000
OK nyc_tlc_yellow     overlap=0  total=1,000,000/1,000,000  authoring=0.2000

All splits valid.


## 9 · Write the provenance recordOne file capturing everything needed to reproduce this notebook's output.

In [ ]:
import json, datetime

def build_provenance():
    return {
        'notebook': 'P2_N0_environment',
        'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'census_version': C.CENSUS_VERSION,
        'prompt_version': P.PROMPT_VERSION,
        'checkpoint_version': CK.CHECKPOINT_VERSION,
        'source_files': file_hashes,
        'read_config': {c: C.CORPUS_READ_CONFIG.get(c, 'external') for c in DATA_PATHS},
        'splits': manifests,
    }

prov, _ = ckpt.step('n0_provenance', 'json', build_provenance,
                    code_version=C.CENSUS_VERSION)

out = ARTIFACTS / 'N0_provenance.json'
out.write_text(json.dumps(prov, indent=2, default=str))
print('Wrote provenance record to:', out)

  [cached] n0_provenance.json  (written 2026-08-08T14:34:13+00:00)
Wrote provenance record to: /content/drive/MyDrive/Paper2_RuleAuthorship/artifacts/N0_provenance.json


## 10 · StatusRe-run this cell any time to see what is already built.

In [ ]:
ckpt.status()
print()
print('N0 complete. Next: N1 — build the census and the A2-A5 prompt payloads.')

Checkpoints in /content/drive/MyDrive/Paper2_RuleAuthorship/checkpoints
  OK  a3_doc_coverage.json                     708 B
  OK  census_bank_marketing.json               15,728 B
  OK  census_diabetes_130us.json               42,999 B
  OK  census_nyc_tlc_yellow.json               21,889 B
  OK  census_online_retail_ii.json             12,491 B
  OK  n0_provenance.json                       4,816 B
  OK  n1_provenance.json                       3,126 B
  OK  n3_rule_corpus.json                      4,172,519 B
  OK  n3_scored_rulesets.json                  164,524 B
  OK  n4_baselines.json                        3,785 B
  OK  n4_downstream_results.json               296,476 B
  OK  prompt_payloads.json                     176,012 B
  OK  source_file_hashes.json                  704 B
  OK  split_bank_marketing.json                486,750 B
  OK  split_diabetes_130us.json                1,110,623 B
  OK  split_nyc_tlc_yellow.json                12,728,159 B
  OK  split_online_retail_i